# ProspectRL — Multi-Agent Training (4-Phase Pipeline)

Train a team of mining turtles using a 4-phase progression:

| Phase | Name | Steps | Agents | Method |
|-------|------|-------|--------|--------|
| 1 | Task Mastery | ~4M | 1 | MaskablePPO + SB3 `model.learn()` |
| 2 | Multi-Agent Execution | ~2M | 4→32 | Frozen policy + heuristic coordinator |
| 3 | Coordinator GNN Training | ~2M | 32 | REINFORCE on GNN, agents frozen |
| 4 | Joint Training | ~3M | 32 | Blended heuristic/GNN + unfrozen agent |

**Phase 1** is the primary training target — it teaches the agent to complete
individual tasks (MOVE_TO, EXCAVATE, MINE_ORE, RETURN_TO) using the multi-agent
observation space and feature extractor.

**Phases 2–4** use manual training loops (not `model.learn()`) and
progressively introduce multi-agent coordination.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

# Clean up ALL previous clone attempts
for old in [Path("/content/prospect_rl"), Path("/content/lib/prospect_rl")]:
    if old.is_symlink():
        old.unlink()
    elif old.exists():
        shutil.rmtree(old)

# Remove stale sys.path entries and cached modules
sys.path = [p for p in sys.path if "prospect_rl" not in p]
for key in list(sys.modules):
    if key.startswith("prospect_rl") or key.startswith("multiagent"):
        del sys.modules[key]

# Clone into /content/lib/prospect_rl
LIB = Path("/content/lib")
PKG = LIB / "prospect_rl"
LIB.mkdir(exist_ok=True)

subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/Mrepp/ProspectRL.git", str(PKG)],
    check=True,
)

# Install dependencies (scipy needed for Hungarian matching in coordinator)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "gymnasium>=0.29.0",
    "stable-baselines3[extra]>=2.3.0",
    "sb3-contrib>=2.3.0",
    "opensimplex>=0.4",
    "numpy>=1.24.0",
    "torch>=2.0.0",
    "tensorboard>=2.14.0",
    "scipy>=1.10.0",
])

sys.path.insert(0, str(LIB))
import prospect_rl.env
import prospect_rl.multiagent
print(f"prospect_rl loaded from {prospect_rl.env.__file__}")
print("Setup complete!")

In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')

# ===================== CONFIGURATION =====================
N_ENVS = 16                    # Parallel environments (Phase 1)
SEED = 42
WORLD_SIZE = (64, 128, 64)     # World dimensions for all phases
DEVICE = "auto"                # "auto", "cuda", or "cpu"

# Phase-specific timesteps (adjust for Colab time limits)
PHASE1_TIMESTEPS = 4_000_000   # Full: 4M, Quick test: 500_000
PHASE2_TIMESTEPS = 2_000_000   # Full: 2M
PHASE3_TIMESTEPS = 2_000_000   # Full: 2M
PHASE4_TIMESTEPS = 3_000_000   # Full: 3M

# Ore preference for Phases 2-4 (None = diamond)
PREFERENCE = None

# Checkpoint paths
BASE_CHECKPOINT_DIR = "/content/drive/MyDrive/prospect_rl_multiagent"
RESUME = True

os.makedirs(BASE_CHECKPOINT_DIR, exist_ok=True)

# ===================== SUMMARY =====================
from prospect_rl.multiagent.config import MultiAgentConfig
ma_config = MultiAgentConfig()

print("=" * 60)
print("  MULTI-AGENT TRAINING CONFIGURATION")
print("=" * 60)
print(f"  World size:      {WORLD_SIZE}")
print(f"  Device:          {DEVICE}")
print(f"  Seed:            {SEED}")
print(f"  Checkpoint dir:  {BASE_CHECKPOINT_DIR}")
print()
print("  Phase Overview:")
print(f"    Phase 1: {PHASE1_TIMESTEPS:>10,} steps | 1 agent  | MaskablePPO")
print(f"    Phase 2: {PHASE2_TIMESTEPS:>10,} steps | 4->32    | Frozen policy")
print(f"    Phase 3: {PHASE3_TIMESTEPS:>10,} steps | 32       | GNN REINFORCE")
print(f"    Phase 4: {PHASE4_TIMESTEPS:>10,} steps | 32       | Joint training")
print(f"  Coordinator K:   {ma_config.coordinator_interval_k}")
print(f"  Mining radius:   {ma_config.mining_radius}")
print("=" * 60)

## Phase 1: Single-Agent Task Mastery

Phase 1 trains the agent policy to complete individual tasks in isolation:

| Task Type | Weight | What the Agent Learns |
|-----------|--------|----------------------|
| MOVE_TO | 35% | Navigate to a target position |
| EXCAVATE | 40% | Clear blocks in a bounding box |
| MINE_ORE | 20% | Mine specific ores (distribution preference) |
| RETURN_TO | 5% | Navigate back to base |

**Difficulty curriculum** (automatic):
- 0–33%: Easy (small boxes, short distances)
- 33–66%: Medium (larger boxes, moderate distances)
- 66–100%: Hard (full-size boxes, long distances)

Uses `Phase1TaskEnv` wrapping `MultiAgentMiningEnv` with `mining_radius=999`
(A* disabled — RL must learn all navigation).

In [ ]:
from prospect_rl.multiagent.training.phase1_env import make_phase1_env

env = make_phase1_env(
    n_envs=N_ENVS,
    world_size=WORLD_SIZE,
    seed=SEED,
    max_episode_steps=1000,
    use_subproc=True,
)

# Show observation and action spaces
print("Observation space:")
for key, space in env.observation_space.spaces.items():
    print(f"  {key:>8}: shape={space.shape}  dtype={space.dtype}")
print(f"\nAction space: Discrete({env.action_space.n})")
print(f"  actions: forward, back, up, down, turn_left, turn_right,")
print(f"           dig, dig_up, dig_down")
print(f"\nVecNormalize: normalizes 'scalars' key only")
print(f"Number of parallel envs: {N_ENVS}")

In [ ]:
import json
from pathlib import Path

from prospect_rl.models.callbacks import DriveCheckpointCallback
from prospect_rl.models.ppo_config import linear_schedule
from prospect_rl.multiagent.agent.agent_policy import MultiAgentFeatureExtractor
from prospect_rl.multiagent.training.phase1_env import (
    Phase1DifficultyCallback,
    Phase1MetricsCallback,
)
from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import CallbackList
from stable_baselines3.common.vec_env import VecNormalize

PHASE1_DIR = Path(BASE_CHECKPOINT_DIR) / "phase1"
PHASE1_DIR.mkdir(parents=True, exist_ok=True)

reset_num_timesteps = True
initial_difficulty = 0

def _create_fresh_model(env):
    return MaskablePPO(
        "MultiInputPolicy",
        env,
        learning_rate=linear_schedule(3e-4),
        n_steps=1024,
        batch_size=2048,
        n_epochs=5,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.15,
        ent_coef=0.10,
        vf_coef=0.75,
        max_grad_norm=0.5,
        normalize_advantage=True,
        policy_kwargs={
            "features_extractor_class": MultiAgentFeatureExtractor,
            "net_arch": {"pi": [64, 64], "vf": [128, 128]},
        },
        seed=SEED,
        tensorboard_log="./tb_logs_multiagent",
        device=DEVICE,
        verbose=1,
    )

if RESUME:
    latest_path = PHASE1_DIR / "latest.json"
    if latest_path.exists():
        manifest = json.loads(latest_path.read_text())
        print(f"Resuming from step {manifest['step']}")

        # Load VecNormalize stats
        env = VecNormalize.load(manifest["vecnormalize_path"], env.venv)
        env.training = True
        env.norm_reward = True

        # Load model
        model = MaskablePPO.load(
            manifest["model_path"],
            env=env,
            device=DEVICE,
        )
        reset_num_timesteps = False

        # Estimate difficulty from saved timestep
        progress = manifest["step"] / PHASE1_TIMESTEPS
        if progress >= 0.66:
            initial_difficulty = 2
        elif progress >= 0.33:
            initial_difficulty = 1
        print(f"  Estimated difficulty level: {initial_difficulty}")
    else:
        print("No checkpoint found, starting fresh")
        model = _create_fresh_model(env)
else:
    model = _create_fresh_model(env)

print(f"\nModel on device: {model.device}")
print(f"Policy architecture:")
print(f"  Features extractor: MultiAgentFeatureExtractor")
print(f"  Pi network: [64, 64]")
print(f"  Vf network: [128, 128]")
print(f"  Output dim: 328 (CNN=256, Scalars=64, Pref=8)")

## Training

**What to watch in TensorBoard:**
- `phase1/task_completion_rate` — overall task success rate, target > 0.6
- `phase1/MOVE_TO_completion_rate` — navigation task success
- `phase1/EXCAVATE_completion_rate` — excavation task success
- `phase1/MINE_ORE_completion_rate` — mining task success (hardest)
- `phase1/tasks_per_episode` — should increase as tasks complete faster
- `phase1/difficulty` — should progress 0 → 1 → 2

Phase 1 with 4M timesteps takes 1–3 hours on a Colab T4 GPU depending on
world size. Start with a smaller run (500K) to verify everything works.

In [ ]:
callbacks = CallbackList([
    DriveCheckpointCallback(
        checkpoint_dir=str(PHASE1_DIR),
        save_freq=25_000,
        max_kept=3,
        verbose=1,
    ),
    Phase1DifficultyCallback(
        initial_difficulty=initial_difficulty,
        verbose=1,
    ),
    Phase1MetricsCallback(verbose=0),
])

model.learn(
    total_timesteps=PHASE1_TIMESTEPS,
    callback=callbacks,
    reset_num_timesteps=reset_num_timesteps,
)

# Save final checkpoint
final_dir = PHASE1_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)
model.save(str(final_dir / "model.zip"))
env.save(str(final_dir / "vecnormalize.pkl"))
print(f"Phase 1 complete! Final model saved to {final_dir}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./tb_logs_multiagent

## Phase 1 Evaluation

Evaluate the trained Phase 1 model by running tasks and measuring per-task-type
completion rates. Uses deterministic policy actions.

In [ ]:
import numpy as np
from collections import defaultdict
from prospect_rl.multiagent.training.phase1_env import Phase1TaskEnv
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Load final model for evaluation
eval_model = MaskablePPO.load(
    str(PHASE1_DIR / "final" / "model.zip"), device=DEVICE,
)

# Create eval env with VecNormalize stats from training
eval_venv = DummyVecEnv([lambda: Phase1TaskEnv(
    world_size=WORLD_SIZE,
    ore_density_multiplier=3.0,
    max_episode_steps=1000,
    seed=SEED + 1000,
    task_step_limit=200,
)])
eval_env = VecNormalize.load(
    str(PHASE1_DIR / "final" / "vecnormalize.pkl"), eval_venv,
)
eval_env.training = False
eval_env.norm_reward = False

N_EVAL_EPISODES = 10
task_results = defaultdict(lambda: {"completed": 0, "total": 0, "steps": []})
episode_rewards = []

for ep in range(N_EVAL_EPISODES):
    obs = eval_env.reset()
    episode_reward = 0.0
    done = False

    while not done:
        masks = eval_env.env_method("action_masks")[0]
        action, _ = eval_model.predict(
            obs, action_masks=np.array([masks]),
            deterministic=True,
        )
        obs, reward, done_arr, info_arr = eval_env.step(action)
        episode_reward += float(reward[0])
        done = done_arr[0]

        info = info_arr[0]
        if info.get("task_ended", False):
            tt = info.get("ended_task_type", "UNKNOWN")
            task_results[tt]["total"] += 1
            if info.get("ended_task_complete", False):
                task_results[tt]["completed"] += 1
            task_results[tt]["steps"].append(
                info.get("ended_task_steps", 0)
            )

    episode_rewards.append(episode_reward)
    print(f"  Episode {ep+1}/{N_EVAL_EPISODES}: reward={episode_reward:.2f}")

# Print results
print(f"\n{'Task Type':<15} {'Rate':>8} {'Completed':>10} "
      f"{'Total':>8} {'Mean Steps':>12}")
print("-" * 58)
for tt in ["MOVE_TO", "EXCAVATE", "MINE_ORE", "RETURN_TO"]:
    r = task_results[tt]
    if r["total"] > 0:
        rate = r["completed"] / r["total"]
        mean_steps = np.mean(r["steps"]) if r["steps"] else 0
        print(f"{tt:<15} {rate:>8.1%} {r['completed']:>10} "
              f"{r['total']:>8} {mean_steps:>12.1f}")

print(f"\nMean episode reward: {np.mean(episode_rewards):.2f} "
      f"(std={np.std(episode_rewards):.2f})")

eval_env.close()

---

## Phase 2: Multi-Agent Execution (Heuristic Coordinator)

Phase 2 loads the Phase 1 trained policy and tests it with multiple agents
in a shared world. The coordinator uses a heuristic (belief map + expected
remaining ore) to assign tasks — no GNN training yet.

**Agent count curriculum:** 4 → 8 → 16 → 32 agents, scaling every 500K steps.

The agent policy is **frozen** (inference only). This phase validates that
the learned behaviors transfer to the multi-agent setting with A* navigation.

In [ ]:
import torch
import numpy as np
from prospect_rl.env.world.world import World
from prospect_rl.multiagent.config import MultiAgentConfig
from prospect_rl.multiagent.training.multi_agent_vec_env import MultiAgentVecEnv

# Load Phase 1 model and freeze
phase1_model = MaskablePPO.load(
    str(PHASE1_DIR / "final" / "model.zip"), device=DEVICE,
)
for param in phase1_model.policy.parameters():
    param.requires_grad_(False)

# Build preference vector
if PREFERENCE is not None:
    preference = np.array(PREFERENCE, dtype=np.float32)
else:
    preference = np.zeros(8, dtype=np.float32)
    preference[3] = 1.0  # Diamond

config = MultiAgentConfig()
task_reward_config = {
    "completion_reward": config.task_completion_reward,
    "progress_reward": config.task_progress_reward,
    "block_cleared_reward": config.block_cleared_reward,
    "regress_penalty": config.regress_penalty,
    "idle_penalty": config.idle_penalty,
    "box_stay_bonus": config.box_stay_bonus,
    "box_leave_penalty": config.box_leave_penalty,
}

def create_vecenv(n_agents):
    world = World(size=WORLD_SIZE, seed=SEED, ore_density_multiplier=3.0)
    return MultiAgentVecEnv(
        world=world,
        n_agents=n_agents,
        preference=preference,
        coordinator_interval_k=config.coordinator_interval_k,
        mining_radius=config.mining_radius,
        seed=SEED,
        task_reward_config=task_reward_config,
        congestion_radius=config.astar_congestion_radius,
        excavate_ore_threshold=config.excavate_ore_threshold,
    )

# Agent count curriculum
schedule = config.agent_count_schedule   # [4, 8, 16, 32]
interval = config.agent_count_step_interval  # 500_000
schedule_idx = 0
current_n = schedule[0]

vec_env = create_vecenv(current_n)
obs_dict, _info = vec_env.reset()

step = 0
total = PHASE2_TIMESTEPS
phase2_dir = Path(BASE_CHECKPOINT_DIR) / "phase2"
phase2_dir.mkdir(parents=True, exist_ok=True)
reward_window = []
LOG_INTERVAL = 10_000
CKPT_INTERVAL = 100_000

print(f"Phase 2: Starting with {current_n} agents, {total:,} total steps")
print(f"  Agent schedule: {schedule} every {interval:,} steps")

while step < total:
    # Agent count scaling
    new_idx = min(len(schedule) - 1, step // interval)
    if new_idx > schedule_idx:
        schedule_idx = new_idx
        current_n = schedule[schedule_idx]
        print(f"\n  Scaling to {current_n} agents at step {step:,}")
        vec_env.close()
        vec_env = create_vecenv(current_n)
        obs_dict, _info = vec_env.reset()

    masks = vec_env.get_action_masks()
    actions, _ = phase1_model.predict(
        obs_dict, action_masks=masks, deterministic=False,
    )
    obs_dict, rewards, terms, truncs, infos = vec_env.step(actions)
    step += vec_env.num_envs
    reward_window.append(float(np.mean(rewards)))

    # Logging
    if step % LOG_INTERVAL < vec_env.num_envs:
        avg_r = np.mean(reward_window[-100:]) if reward_window else 0
        print(f"  Step {step:>10,} | agents={current_n:>3} | "
              f"avg_reward={avg_r:.4f}")

    # Checkpointing
    if step % CKPT_INTERVAL < vec_env.num_envs:
        ckpt_path = phase2_dir / f"step_{step}"
        ckpt_path.mkdir(exist_ok=True)
        torch.save(
            vec_env._coordinator._gnn.state_dict(),
            str(ckpt_path / "coordinator.pt"),
        )
        manifest = {"step": step, "n_agents": current_n, "phase": 2}
        (ckpt_path / "manifest.json").write_text(json.dumps(manifest))

vec_env.close()
print(f"\nPhase 2 complete at step {step:,}")

## Phase 3: Coordinator GNN Training (REINFORCE)

Phase 3 freezes the agent policy and trains the GNN coordinator via REINFORCE
on team-level reward. The coordinator runs every K=50 steps and assigns agents
to regions using Hungarian matching on GNN-predicted scores.

**Note:** On Colab without `torch-geometric`, this uses `CoordinatorGNNSimple`
(MLP-based score prediction) instead of the full GATv2Conv GNN.

In [ ]:
from prospect_rl.multiagent.training.coordinator_trainer import CoordinatorTrainer

# Load and freeze Phase 1 model
phase3_model = MaskablePPO.load(
    str(PHASE1_DIR / "final" / "model.zip"), device=DEVICE,
)
for param in phase3_model.policy.parameters():
    param.requires_grad_(False)

# Create env with 32 agents
N_AGENTS_P3 = 32
vec_env = create_vecenv(N_AGENTS_P3)

# Create REINFORCE trainer for GNN
trainer = CoordinatorTrainer(
    gnn=vec_env._coordinator._gnn,
    lr=config.coordinator_lr,
    ent_coef=config.coordinator_ent_coef,
)

obs_dict, _info = vec_env.reset()
step = 0
total = PHASE3_TIMESTEPS
window_reward = 0.0
phase3_dir = Path(BASE_CHECKPOINT_DIR) / "phase3"
phase3_dir.mkdir(parents=True, exist_ok=True)

print(f"Phase 3: GNN training with {N_AGENTS_P3} agents, {total:,} steps")
print(f"  Coordinator LR: {config.coordinator_lr}")
print(f"  Entropy coef: {config.coordinator_ent_coef}")

while step < total:
    masks = vec_env.get_action_masks()
    actions, _ = phase3_model.predict(
        obs_dict, action_masks=masks, deterministic=False,
    )
    obs_dict, rewards, terms, truncs, infos = vec_env.step(actions)
    window_reward += float(np.sum(rewards))
    step += vec_env.num_envs

    # Train GNN on coordinator replan events
    coordinator = vec_env._coordinator
    if (coordinator.step_since_replan == 0
            and coordinator._last_x_dict is not None):
        metrics = trainer.update(
            x_dict=coordinator._last_x_dict,
            edge_index_dict=coordinator._last_edge_index_dict,
            row_idx=coordinator._last_row_idx,
            col_idx=coordinator._last_col_idx,
            team_reward=window_reward,
        )
        window_reward = 0.0

    # Logging
    if step % LOG_INTERVAL < vec_env.num_envs:
        avg_r = float(np.mean(rewards))
        log_msg = f"  Step {step:>10,} | avg_reward={avg_r:.4f}"
        if trainer.total_updates > 0:
            log_msg += (f" | GNN loss={trainer._last_loss:.4f}"
                       f" | entropy={trainer._last_entropy:.4f}"
                       f" | updates={trainer.total_updates}")
        print(log_msg)

    # Checkpointing
    if step % CKPT_INTERVAL < vec_env.num_envs:
        ckpt_path = phase3_dir / f"step_{step}"
        ckpt_path.mkdir(exist_ok=True)
        torch.save(
            coordinator._gnn.state_dict(),
            str(ckpt_path / "coordinator.pt"),
        )
        manifest = {
            "step": step, "n_agents": N_AGENTS_P3, "phase": 3,
            "gnn_updates": trainer.total_updates,
        }
        (ckpt_path / "manifest.json").write_text(json.dumps(manifest))

vec_env.close()
print(f"\nPhase 3 complete: {step:,} steps, {trainer.total_updates} GNN updates")

## Phase 4: Joint Training (Blended Assignment)

Phase 4 trains both the agent policy and GNN coordinator jointly. Uses a
blended assignment that interpolates between heuristic and GNN scores:

| Progress | Alpha | Meaning |
|----------|-------|---------|
| 0–25% | 0.1 | 90% heuristic, 10% GNN |
| 25–50% | 0.3 | 70% heuristic, 30% GNN |
| 50–75% | 0.6 | 40% heuristic, 60% GNN |
| 75–100% | 1.0 | Pure GNN |

**Note:** The agent policy is unfrozen but the current implementation uses
a manual predict/step loop. For full PPO gradient updates on the agent,
use `train_multiagent.py --phase 4`.

In [ ]:
# Load agent model (now TRAINABLE, not frozen)
phase4_model = MaskablePPO.load(
    str(PHASE1_DIR / "final" / "model.zip"), device=DEVICE,
)

# Create env
N_AGENTS_P4 = 32
vec_env = create_vecenv(N_AGENTS_P4)

# Try to load Phase 3 coordinator checkpoint
phase3_ckpts = sorted(phase3_dir.glob("step_*/coordinator.pt"))
if phase3_ckpts:
    latest_ckpt = phase3_ckpts[-1]
    vec_env._coordinator._gnn.load_state_dict(
        torch.load(str(latest_ckpt), map_location=DEVICE)
    )
    print(f"Loaded Phase 3 coordinator from {latest_ckpt}")
else:
    print("No Phase 3 checkpoint found, using untrained coordinator")

# Create GNN trainer
trainer = CoordinatorTrainer(
    gnn=vec_env._coordinator._gnn,
    lr=config.coordinator_lr,
    ent_coef=config.coordinator_ent_coef,
)

# Alpha schedule
alpha_schedule = config.blend_alpha_schedule   # [0.1, 0.3, 0.6, 1.0]
alpha_interval = config.blend_alpha_step_interval  # 750_000
alpha_idx = 0
vec_env._coordinator.alpha = alpha_schedule[0]

obs_dict, _info = vec_env.reset()
step = 0
total = PHASE4_TIMESTEPS
window_reward = 0.0
phase4_dir = Path(BASE_CHECKPOINT_DIR) / "phase4"
phase4_dir.mkdir(parents=True, exist_ok=True)

print(f"Phase 4: Joint training, {N_AGENTS_P4} agents, {total:,} steps")
print(f"  Alpha schedule: {alpha_schedule} every {alpha_interval:,} steps")

while step < total:
    # Update alpha schedule
    new_idx = min(len(alpha_schedule) - 1, step // alpha_interval)
    if new_idx > alpha_idx:
        alpha_idx = new_idx
        vec_env._coordinator.alpha = alpha_schedule[alpha_idx]
        print(f"\n  Alpha -> {alpha_schedule[alpha_idx]:.2f} at step {step:,}")

    masks = vec_env.get_action_masks()
    actions, _ = phase4_model.predict(
        obs_dict, action_masks=masks, deterministic=False,
    )
    obs_dict, rewards, terms, truncs, infos = vec_env.step(actions)
    window_reward += float(np.sum(rewards))
    step += vec_env.num_envs

    # Train GNN on coordinator replan
    coordinator = vec_env._coordinator
    if (coordinator.step_since_replan == 0
            and coordinator._last_x_dict is not None):
        trainer.update(
            x_dict=coordinator._last_x_dict,
            edge_index_dict=coordinator._last_edge_index_dict,
            row_idx=coordinator._last_row_idx,
            col_idx=coordinator._last_col_idx,
            team_reward=window_reward,
        )
        window_reward = 0.0

    # Logging
    if step % LOG_INTERVAL < vec_env.num_envs:
        avg_r = float(np.mean(rewards))
        print(f"  Step {step:>10,} | alpha={alpha_schedule[alpha_idx]:.2f} | "
              f"avg_reward={avg_r:.4f}")

    # Checkpointing
    if step % CKPT_INTERVAL < vec_env.num_envs:
        ckpt_path = phase4_dir / f"step_{step}"
        ckpt_path.mkdir(exist_ok=True)
        torch.save(
            coordinator._gnn.state_dict(),
            str(ckpt_path / "coordinator.pt"),
        )
        manifest = {
            "step": step, "n_agents": N_AGENTS_P4, "phase": 4,
            "alpha": alpha_schedule[alpha_idx],
        }
        (ckpt_path / "manifest.json").write_text(json.dumps(manifest))

vec_env.close()
print(f"\nPhase 4 complete at step {step:,}")

## Summary

**Phase 1** (Task Mastery) is the critical training phase. The agent must learn
to reliably complete MOVE_TO, EXCAVATE, MINE_ORE, and RETURN_TO tasks before
multi-agent coordination adds value.

**Phases 2–4** demonstrate the multi-agent pipeline on manual loops.
For production use, see `train_multiagent.py` which provides the same logic
with CLI arguments and logging.

### Checkpoints saved to Google Drive

```
prospect_rl_multiagent/
  phase1/
    latest.json              # Resume manifest
    step_*/model.zip         # MaskablePPO checkpoint
    step_*/vecnormalize.pkl
    final/model.zip          # Final Phase 1 model
  phase2/
    step_*/coordinator.pt
  phase3/
    step_*/coordinator.pt    # Trained GNN
  phase4/
    step_*/coordinator.pt    # Jointly trained GNN
```

### To deploy the trained system

1. Use `phase1/final/model.zip` as the agent policy
2. Use the latest `phase3/` or `phase4/` `coordinator.pt` as the GNN
3. Load both into `MultiAgentVecEnv` for inference